# Multiclass Classifier Seed07 Workflow

This notebook evaluates the single `seed07` multiclass classifier checkpoint for the full labeled `all_80_10_10` workflow and can also produce unlabeled pseudo-label outputs from that run.

Recommended order:
- train the `seed07` full-labeled run with this branch's local seed-specific config
- confirm the run produced a `best_model.pt`, `metrics.json`, and `test_predictions.csv`
- run this notebook to review held-out test results for `seed07`
- optionally generate unlabeled pseudo-label outputs into this notebook branch's local artifact folder

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / "src" / "wafer_defect").exists() and (candidate / "configs").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise RuntimeError("Could not locate repo root containing src/wafer_defect and configs/")

SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

NOTEBOOK_DIR = REPO_ROOT / "experiments/classifier/multiclass/x64/seed07"

from wafer_defect.config import load_toml

In [ ]:
data_config_path = NOTEBOOK_DIR / "data_config.toml"
train_config_path = NOTEBOOK_DIR / "train_config.toml"
data_config = load_toml(data_config_path)
train_config = load_toml(train_config_path)

seed07_artifact_dir = NOTEBOOK_DIR / "artifacts" / Path(train_config["training"]["output_dir"]).name
checkpoint_path = seed07_artifact_dir / "best_model.pt"
history_path = seed07_artifact_dir / "history.csv"
metrics_path = seed07_artifact_dir / "metrics.json"
test_predictions_path = seed07_artifact_dir / "test_predictions.csv"
unlabeled_predictions_path = seed07_artifact_dir / "unlabeled_predictions.csv"
confidence_threshold = 0.98
RUN_PSEUDOLABEL_INFERENCE = False

required_paths = [checkpoint_path, metrics_path, test_predictions_path]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Missing required seed07 classifier artifacts:\n" + "\n".join(str(path) for path in missing_paths)
    )

print("Data config:", data_config_path)
print("Seed07 config:", train_config_path)
print("Seed07 artifact dir:", seed07_artifact_dir)
print("Checkpoint:", checkpoint_path)
print("Unlabeled predictions path:", unlabeled_predictions_path)
print("Confidence threshold:", confidence_threshold)
print("Run pseudo-label inference:", RUN_PSEUDOLABEL_INFERENCE)

In [ ]:
seed07_artifact_dir.mkdir(parents=True, exist_ok=True)

if RUN_PSEUDOLABEL_INFERENCE:
    predict_command = [
        sys.executable,
        str(REPO_ROOT / "scripts/classifier/predict_unlabeled_multiclass.py"),
        "--config",
        str(data_config_path),
        "--checkpoint",
        str(checkpoint_path),
        "--output-csv",
        str(unlabeled_predictions_path),
        "--min-confidence",
        str(confidence_threshold),
    ]
    print("Running:", " ".join(predict_command))
    subprocess.run(predict_command, cwd=REPO_ROOT, check=True)
else:
    print(f"RUN_PSEUDOLABEL_INFERENCE is False. Reusing saved predictions from {unlabeled_predictions_path}")

In [ ]:
history = pd.read_csv(history_path) if history_path.exists() else None
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
test_predictions = pd.read_csv(test_predictions_path)
unlabeled_predictions = pd.read_csv(unlabeled_predictions_path) if unlabeled_predictions_path.exists() else None

print("Test accuracy:", metrics["test"]["accuracy"])
print("Test balanced accuracy:", metrics["test"]["balanced_accuracy"])
if history is not None:
    print("Best validation balanced accuracy:", history["val_balanced_accuracy"].max())
if unlabeled_predictions is not None:
    print("Accepted pseudo-label fraction:", unlabeled_predictions["accepted_for_pseudo_label"].mean())
else:
    print("No saved unlabeled predictions found yet. Turn RUN_PSEUDOLABEL_INFERENCE to True to generate them.")

display(test_predictions.head())
if unlabeled_predictions is not None:
    display(unlabeled_predictions.head())